In [0]:
#1 Importing Tools 
import openpyxl
import pandas as pd


from pyspark.sql import functions as F
from datetime import datetime
from openpyxl.styles import NamedStyle

In [0]:
#2 Reduce risk of a timeout by increasing limit to 30 minutes
spark.conf.set("spark.databricks.execution.timeout", "1800")

In [0]:
#3 Loading the master hierarchies table from the lake mart (via ADLS path)

# Set connection settings for prov ref data
analysisLake = "udalstdatacuratedprod.dfs.core.windows.net"
analysisContainer = "reporting"
provFile = "/unrestricted/reference/UKHD/ODS/Provider_Hierarchies_ICB"

provPath = f"abfss://{analysisContainer}@{analysisLake}{provFile}"

# Load provider reference data
df_master_hierarchies = (
    spark.read
         .option("header", "true")
         .option("recursiveFileLookup", "true")
         .parquet(provPath)
)

# Display sample
display(df_master_hierarchies.limit(10))

# Count active records (where Effective_To is NULL)
active_count = df_master_hierarchies.filter(F.col("Effective_To").isNull()).count()

print(f"Number of rows in master hierarchies (active records): {active_count}")

In [0]:
#4 loading ICB to Region mapping table
df_icb_region = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/EROC_ICB_Region_DisplayNames.csv")  

#display(df_icb_region.limit(10))
print(f"Number of rows in icb_region: {df_icb_region.count()}")

In [0]:
#5 loading list of merged providers
df_merged_providers = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/EROC_Merged_Providers.csv")

#display(df_merged_providers.limit(10))
print(f"Number of rows in merged providers: {df_merged_providers.count()}")

In [0]:
#6 creating new provider code from the provider mapping table
provider_code_mapping = df_merged_providers = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/EROC/EROC_Merged_Providers.csv")

#display(df_merged_providers.limit(10))
print(f"Number of rows in merged providers: {df_merged_providers.count()}")

In [0]:
#7 Importing the MHS details

#7a importing MHS metric list and internal ID
mhs_metric_list = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/MHS")

#display(mhs_metric_list.limit(10))
print(f"Number of rows in mhs_metric_list: {mhs_metric_list.count()}")

#7b importing MHS allowable org codes
mhs_allowable_orgs = spark.read.option("header", "true").csv("abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/EROC_Collection/MHS/Allowable_Org_Codes_Status.csv")

#display(mhs_allowable_orgs.limit(10))
print(f"Number of rows in mhs_allowable_orgs: {mhs_allowable_orgs.count()}")

In [0]:
#8 Loading the core monthly snapshot data
from pyspark.sql import functions as F
df_op_activity_snapshot = spark.read.option("header", "true").option("recursiveFileLookup", "true").parquet(
    "abfss://reporting@udalstdatacuratedprod.dfs.core.windows.net/restricted/patientlevel/MESH/OPA/OPA_Core_Monthly_Snapshot/Published/1"
)
display(df_op_activity_snapshot.limit(10))

# Show number of rows in the raw data
row_count = df_op_activity_snapshot.count()
print(f"Number of rows in raw data: {row_count}")

In [0]:
#9 Creating the wide table & inserting new columns for merged providers, with new merger codes and mapping to ICB and Region codes
from pyspark.sql.functions import (
    when,
    col,
    lit,
    create_map,
    coalesce,
    last_day,
    to_date,
    concat_ws,
    current_date,
    add_months,
    date_format
)
import pyspark.sql.functions as F

# 1. Define valid treatment function codes
VALID_TREATMENT_CODES = [
    '100', '101', '102', '104', '105', '106', '108', '110', '111', '115', '120', '130', '140',
    '144', '145', '301', '302', '303', '307', '320', '330', '340', '361', '400', '410', '420',
    '430', '501', '502', '560', '650'
]

# 2. Add Treatment_Function_Code_New (valid vs Other)
opa_with_tfc = df_op_activity_snapshot.withColumn(
    "Treatment_Function_Code_New",
    when(col("Treatment_Function_Code").isin(VALID_TREATMENT_CODES),
         col("Treatment_Function_Code")
    ).otherwise("Other")
)

# 3. Add Treatment_Function_Group (GS / OMFS / T&O / code)
opa_with_groups = opa_with_tfc.withColumn(
    "Treatment_Function_Group",
    when(col("Treatment_Function_Code_New").isin("100", "102", "104", "105", "106"), "GS")
    .when(col("Treatment_Function_Code_New").isin("140", "144", "145"), "OMFS")
    .when(col("Treatment_Function_Code_New").isin("110", "111", "115"), "T&O")
    .otherwise(col("Treatment_Function_Code_New"))
)

# 4. Filter dataset (years, admin category, TFC, attendance)
opa_filtered = opa_with_groups.filter(
    (col("Der_Financial_Year").isin("2023/24", "2024/25", "2025/26")) &  # Update manually if needed
    (col("Administrative_Category") == "01") &
    (col("Treatment_Function_Code") != "812") &
    (col("First_Attendance").isin("1", "2", "3", "4"))
)

# Keep ALL months up to last full month (excludes future months)
last_full_month_yyyymm = date_format(add_months(current_date(), -1), "yyyyMM")

# reinforces start month (April 2023 = 202304) 
start_month_yyyymm = F.lit("202304")  #change if needed

opa_filtered = opa_filtered.filter(
    (col("Der_Activity_Month") >= start_month_yyyymm) &
    (col("Der_Activity_Month") <= last_full_month_yyyymm)
)

# 5. Aggregate metrics by month, provider, and TFC group
opa_agg = opa_filtered.groupBy(
    "Der_Activity_Month",
    "Der_Provider_Code",
    "Treatment_Function_Group"
).agg(
    # All metrics
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_Total"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "3")), 1
        ).otherwise(0)
    ).alias("All_First"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU"),

    F.sum(
        when(
            (col("Der_Number_Procedure") > 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_Proc"),

    F.sum(
        when(
            (col("Der_Number_Procedure") == 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_NoProc"),

    F.sum(
        when(
            (col("Der_Number_Procedure") > 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU_Proc"),

    F.sum(
        when(
            (col("Der_Number_Procedure") == 0) &
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU_NoProc"),

    # Face-to-face
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("1", "2")), 1
        ).otherwise(0)
    ).alias("F2F_Total"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "1"), 1
        ).otherwise(0)
    ).alias("F2F_First"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "2"), 1
        ).otherwise(0)
    ).alias("F2F_FU"),

    # Remote
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance").isin("3", "4")), 1
        ).otherwise(0)
    ).alias("Remote_Total"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "3"), 1
        ).otherwise(0)
    ).alias("Remote_First"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6")) &
            (col("First_Attendance") == "4"), 1
        ).otherwise(0)
    ).alias("Remote_FU"),

    # Did not attends (DNAs)
    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "2", "3", "4")), 1
        ).otherwise(0)
    ).alias("All_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "3")), 1
        ).otherwise(0)
    ).alias("All_First_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("2", "4")), 1
        ).otherwise(0)
    ).alias("All_FU_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "2")), 1
        ).otherwise(0)
    ).alias("F2F_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("3", "4")), 1
        ).otherwise(0)
    ).alias("Remote_DNA"),

    # 2WW DNA
    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "2", "3", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_2WW_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("1", "3")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_First_2WW_DNA"),

    F.sum(
        when(
            (col("Attendance_Status").isin("3", "7")) &
            (col("First_Attendance").isin("2", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_FU_2WW_DNA"),

    # All 2WW appointments
    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6", "3", "7")) &
            (col("First_Attendance").isin("1", "2", "3", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_2WW"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6", "3", "7")) &
            (col("First_Attendance").isin("1", "3")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_First_2WW"),

    F.sum(
        when(
            (col("Attendance_Status").isin("5", "6", "3", "7")) &
            (col("First_Attendance").isin("2", "4")) &
            (col("Priority_Type") == "3"), 1
        ).otherwise(0)
    ).alias("All_FU_2WW")
)

# 6. Add "All" TFC totals by month and provider
METRIC_COLS = [
    c for c in opa_agg.columns
    if c not in ["Der_Activity_Month", "Der_Provider_Code", "Treatment_Function_Group"]
]

opa_all_tfc = opa_agg.groupBy("Der_Activity_Month", "Der_Provider_Code").agg(
    *[F.sum(col(c)).alias(c) for c in METRIC_COLS]
).withColumn("Treatment_Function_Group", lit("All"))

opa_final = opa_agg.unionByName(opa_all_tfc)

# Order by results
opa_final_ordered = opa_final.orderBy("Der_Activity_Month", "Der_Provider_Code", "Treatment_Function_Group")

# 7. Normalise provider codes > Provider_Base_Code. (replicates the SQL logic)
opa_final_ordered = opa_final_ordered.withColumn(
    "Provider_Base_Code",
    when((
        col("Der_Provider_Code").substr(4, 2) == "00") & (col("Der_Provider_Code") != "NQT00"),   # e.g. ABC00
        col("Der_Provider_Code").substr(1, 3)                                                     # -> ABC
    ).otherwise(
        col("Der_Provider_Code")                                                                  # everything else is unchanged
    )
)

# 8. Build mapping from df_merged_providers (Old -> New)
provider_code_mapping_dict = {
    row["Old_Provider_Code"]: row["New_Provider_Code"]
    for row in df_merged_providers.select("Old_Provider_Code", "New_Provider_Code").distinct().collect()
}

# where k, v is present it is just short hand and means key value
mapping_list = []
for k, v in provider_code_mapping_dict.items():
    mapping_list.append(lit(k))
    mapping_list.append(lit(v))

mapping_expr = create_map(*mapping_list)

# 9. Add "Adj Org Code" based on Provider_Base_Code & mapping
opa_final_ordered_with_adj = opa_final_ordered.withColumn(
    "Adj Org Code",
    coalesce(mapping_expr.getItem(col("Provider_Base_Code")), col("Provider_Base_Code"))
)

# 10. Build Org_Code_For_Join to match hierarchy
opa_with_join_code = opa_final_ordered_with_adj.withColumn(
    "Org_Code_For_Join",
    when(
        col("Adj Org Code") == "NQT00",
        col("Adj Org Code")
    ).when(
        (F.length(col("Adj Org Code")) == 5) & col("Adj Org Code").endswith("00"),
        col("Adj Org Code").substr(1, 3)
    ).otherwise(col("Adj Org Code"))
)

# 11. Join to master hierarchies to get ICB (ICB_Code)
opa_with_icb = opa_with_join_code.join(
    df_master_hierarchies.select(
        F.col("Organisation_Code").alias("join_org_code"),
        F.col("ICB_Code").alias("ICB")
    ),
    opa_with_join_code["Org_Code_For_Join"] == F.col("join_org_code"),
    "left"
).drop("join_org_code")

# 12. Join to df_icb_region to get Region_Code
opa_with_icb_region = opa_with_icb.join(
    df_icb_region.select(
        F.col("ICB_Code").alias("join_icb"),
        F.col("Region_Code")
    ),
    opa_with_icb["ICB"] == F.col("join_icb"),
    "left"
).drop("join_icb")

# 13. Build Der_Activity_Month_Date (YYYYMM -> month end date)
opa_with_icb_region = opa_with_icb_region.withColumn(
    "Der_Activity_Month_Date",
    last_day(
        to_date(
            concat_ws(
                "-",
                col("Der_Activity_Month").substr(1, 4),
                col("Der_Activity_Month").substr(5, 2),
                lit("01")
            )
        )
    )
)

# 14. Aggregate to final level & sort
id_cols = ["Der_Activity_Month_Date", "Treatment_Function_Group", "Region_Code", "ICB", "Adj Org Code"]

metric_cols = [
    c for c in opa_with_icb_region.columns
    if c not in id_cols + [
        "Der_Activity_Month",
        "Der_Provider_Code",
        "Treatment_Function_Code_New",
        "Provider_Base_Code",
        "Org_Code_For_Join"
    ]
]

opa_final_processed = (
    opa_with_icb_region
    .groupBy(*[F.col(c) for c in id_cols])
    .agg(*[F.sum(F.col(c)).alias(c) for c in metric_cols])
    .orderBy("Der_Activity_Month_Date", "Region_Code", "ICB", "Adj Org Code", "Treatment_Function_Group")
)

display(opa_final_processed.limit(10))
print(f"opa_final_processed: {opa_final_processed.count()}")


In [0]:
#10 — Safe metric calculation (robust to missing columns) | adding all the extra metrics to the wide table
from pyspark.sql import functions as F

df = df = opa_final_processed 

def safe_add(df, new_col, expr_fn, required_cols):
    if all(c in df.columns for c in required_cols):
        return df.withColumn(new_col, expr_fn(df))
    else:
        return df.withColumn(new_col, F.lit(None))

metrics = [
    ("All_DNA_Over_All_Total", lambda d: F.when(
        (F.col("All_Total") + F.col("All_DNA")) != 0,
        (F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
    ), ["All_Total", "All_DNA"]),
    ("All_DNA_Over_All_Total_IG", lambda d: F.when(
        (F.col("All_Total") + F.col("All_DNA")) != 0,
        (F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
    ), ["All_Total", "All_DNA"]),
    ("All_First_DNA_Over_All_First", lambda d: F.when(
        (F.col("All_First") + F.col("All_First_DNA")) != 0,
        (F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100
    ), ["All_First", "All_First_DNA"]),
    ("All_First_DNA_Over_All_First_IG", lambda d: F.when(
        (F.col("All_First") + F.col("All_First_DNA")) != 0,
        (F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100
    ), ["All_First", "All_First_DNA"]),
    ("All_FU_DNA_Over_All_FU", lambda d: F.when(
        (F.col("All_FU") + F.col("All_FU_DNA")) != 0,
        (F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100
    ), ["All_FU", "All_FU_DNA"]),
    ("All_FU_DNA_Over_All_FU_IG", lambda d: F.when(
        (F.col("All_FU") + F.col("All_FU_DNA")) != 0,
        (F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100
    ), ["All_FU", "All_FU_DNA"]),
    ("All_2WW_DNA_Over_All_2WW", lambda d: F.when(
        (F.col("All_2WW") != 0) & (F.col("All_2WW").isNotNull()),
        (F.col("All_2WW_DNA") / F.col("All_2WW")) * 100
    ), ["All_2WW_DNA", "All_2WW"]),
    ("All_FU_2WW_DNA_Over_All_FU_2WW", lambda d: F.when(
        (F.col("All_FU_2WW") != 0) & (F.col("All_FU_2WW").isNotNull()),
        (F.col("All_FU_2WW_DNA") / F.col("All_FU_2WW")) * 100
    ), ["All_FU_2WW_DNA", "All_FU_2WW"]),
    ("All_First_2WW_DNA_Over_All_First_2WW", lambda d: F.when(
        (F.col("All_First_2WW") != 0) & (F.col("All_First_2WW").isNotNull()),
        (F.col("All_First_2WW_DNA") / F.col("All_First_2WW")) * 100
    ), ["All_First_2WW_DNA", "All_First_2WW"]),
    ("All_FU_NoProc_Over_All_FU", lambda d: F.when(
        F.col("All_FU") != 0, (F.col("All_FU_NoProc") / F.col("All_FU")) * 100
    ), ["All_FU_NoProc", "All_FU"]),
    ("All_FU_Proc_Over_All_FU", lambda d: F.when(
        F.col("All_FU") != 0, (F.col("All_FU_Proc") / F.col("All_FU")) * 100
    ), ["All_FU_Proc", "All_FU"]),
    ("All_FU_To_All_First", lambda d: F.when(
        F.col("All_First") != 0, (F.col("All_FU") / F.col("All_First"))
    ), ["All_FU", "All_First"]),
    ("All_FU_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_FU") / F.col("All_Total")) * 100
    ), ["All_FU", "All_Total"]),
    ("All_First_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_First") / F.col("All_Total")) * 100
    ), ["All_First", "All_Total"]),
    ("All_NoProc_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_NoProc") / F.col("All_Total")) * 100
    ), ["All_NoProc", "All_Total"]),
    ("All_Proc_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("All_Proc") / F.col("All_Total")) * 100
    ), ["All_Proc", "All_Total"]),
    ("Remote_Total_Over_All_Total", lambda d: F.when(
        F.col("All_Total") != 0, (F.col("Remote_Total") / F.col("All_Total")) * 100
    ), ["Remote_Total", "All_Total"]),
    ("Remote_FU_Over_All_FU", lambda d: F.when(
        F.col("All_FU") != 0, (F.col("Remote_FU") / F.col("All_FU")) * 100
    ), ["Remote_FU", "All_FU"]),
    ("Remote_First_Over_All_First", lambda d: F.when(
        F.col("All_First") != 0, (F.col("Remote_First") / F.col("All_First")) * 100
    ), ["Remote_First", "All_First"]),
    ("F2F_DNA_Over_F2F_Total", lambda d: F.when(
        (F.col("F2F_Total") + F.col("F2F_DNA")) != 0,
        (F.col("F2F_DNA") / (F.col("F2F_Total") + F.col("F2F_DNA"))) * 100
    ), ["F2F_Total", "F2F_DNA"]),
    ("Remote_DNA_Over_Remote_Total", lambda d: F.when(
        (F.col("Remote_Total") + F.col("Remote_DNA")) != 0,
        (F.col("Remote_DNA") / (F.col("Remote_Total") + F.col("Remote_DNA"))) * 100
    ), ["Remote_Total", "Remote_DNA"]),
]

for name, expr, req in metrics:
    df = safe_add(df, name, expr, req)

simple_copies = [
    ("All_DNA1", "All_DNA"),
    ("All_DNA2", "All_DNA"),
    ("All_First1", "All_First"),
    ("All_First2", "All_First"),
    ("All_First3", "All_First"),
    ("All_FU1", "All_FU"),
    ("All_FU2", "All_FU"),
    ("All_FU3", "All_FU"),
    ("All_FU4", "All_FU"),
    ("All_FU5", "All_FU"),
    ("All_Total1", "All_Total"),
    ("All_Total2", "All_Total"),
    ("All_Total3", "All_Total"),
    ("All_Total4", "All_Total"),
    ("All_Total5", "All_Total"),
    ("All_Total6", "All_Total"),
    ("Remote_Total1", "Remote_Total"),
    ("Remote_Total2", "Remote_Total"),
]
for newc, base in simple_copies:
    if base in df.columns:
        df = df.withColumn(newc, F.col(base))
    else:
        df = df.withColumn(newc, F.lit(None))

combos = [
    ("All_First_plus_All_First_DNA", ["All_First", "All_First_DNA"]),
    ("All_FU_plus_All_FU_DNA", ["All_FU", "All_FU_DNA"]),
    ("All_Total_plus_All_DNA", ["All_Total", "All_DNA"]),
    ("F2F_Total_plus_F2F_DNA", ["F2F_Total", "F2F_DNA"]),
    ("Remote_Total_plus_Remote_DNA", ["Remote_Total", "Remote_DNA"]),
]
for newc, cols in combos:
    if all(c in df.columns for c in cols):
        df = df.withColumn(newc, F.col(cols[0]).cast("long") + F.col(cols[1]).cast("long"))
    else:
        df = df.withColumn(newc, F.lit(None))

cols_to_drop = [c for c in ["Der_Activity_Month"] if c in df.columns]
opa_final_with_added_metrics = df.drop(*cols_to_drop)

display(opa_final_with_added_metrics.limit(10))
print(f"opa_final_with_added_metrics: {opa_final_with_added_metrics.count()} rows")

In [0]:
# 11 reshapes the wide outpatient dataset into a long (tidy) format for easier analysis
from pyspark.sql.functions import col, explode, array, struct, lit, concat_ws

# ID columns to keep
id_cols = [
    "Der_Activity_Month_Date",
    "Treatment_Function_Group",
    "Adj Org Code",
    "ICB",
    "Region_Code"
]

# Identify all metric columns
metric_cols = [c for c in opa_final_with_added_metrics.columns if c not in id_cols]

# Unpivot all metrics, casting values to string to avoid type errors
opa_long = (
    opa_final_with_added_metrics
    .select(
        *id_cols,
        explode(array(*[
            struct(
                lit(c).alias("Metric_Name"),
                col(c).cast("string").alias("Metric_Value")
            ) for c in metric_cols
        ])).alias("kv")
    )
    .select(
        *id_cols,
        col("kv.Metric_Name"),
        col("kv.Metric_Value")
    )
)

# Create the combined metric name
opa_long = opa_long.withColumn(
    "Metric_Name_Treatment_Function_Group",
    concat_ws("_", col("Metric_Name"), col("Treatment_Function_Group"))
)

# Order by date
opa_long_ordered = opa_long.orderBy("Der_Activity_Month_Date")

display(opa_long_ordered.limit(10))
#print(f"Number of rows in opa_long_ordered: {opa_long_ordered.count()}"))

In [0]:

# Container 12 — Aggregation for Org / ICB / Region (all sectors)
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit

# 1. Start from Org-level wide dataset from Container 10

df_org = opa_final_with_added_metrics.withColumnRenamed("Adj Org Code", "Adj_Org_Code")

# 2. (Optional) Attach sector type from df_master_hierarchies - Kept for reference / QA, not used in rollups
   
sector_lookup = (
    df_master_hierarchies
    .select(
        col("Organisation_Code").alias("ODS_Code"),
        col("ODS_Organisation_Type")
    )
    .distinct()
)

df_org = (
    df_org
    .join(
        sector_lookup,
        df_org["Adj_Org_Code"] == col("ODS_Code"),
        "left"
    )
    .drop("ODS_Code")
)

# 3. Metric columns to sum at ICB/Region

BASE_COUNT_COLS = [
    "All_Total","All_First","All_FU","All_Proc","All_NoProc",
    "All_FU_Proc","All_FU_NoProc",
    "F2F_Total","F2F_First","F2F_FU","F2F_DNA",
    "Remote_Total","Remote_First","Remote_FU","Remote_DNA",
    "All_DNA","All_First_DNA","All_FU_DNA",
    "All_2WW","All_First_2WW","All_FU_2WW",
    "All_2WW_DNA","All_First_2WW_DNA","All_FU_2WW_DNA"
]

# Small safety feature : only sum columns that actually exist
count_cols = [c for c in BASE_COUNT_COLS if c in df_org.columns]

# 4. Function to add rate metrics

def add_rate_metrics(df):
    return (
        df
        # DNA metrics
        .withColumn("All_DNA_Over_All_Total", F.when(
            (F.col("All_Total") + F.col("All_DNA")) != 0,
            (F.col("All_DNA") / (F.col("All_Total") + F.col("All_DNA"))) * 100
        ))
        .withColumn("All_DNA_Over_All_Total_IG", F.col("All_DNA_Over_All_Total"))
        .withColumn("All_First_DNA_Over_All_First", F.when(
            (F.col("All_First") + F.col("All_First_DNA")) != 0,
            (F.col("All_First_DNA") / (F.col("All_First") + F.col("All_First_DNA"))) * 100
        ))
        .withColumn("All_First_DNA_Over_All_First_IG", F.col("All_First_DNA_Over_All_First"))
        .withColumn("All_FU_DNA_Over_All_FU", F.when(
            (F.col("All_FU") + F.col("All_FU_DNA")) != 0,
            (F.col("All_FU_DNA") / (F.col("All_FU") + F.col("All_FU_DNA"))) * 100
        ))
        .withColumn("All_FU_DNA_Over_All_FU_IG", F.col("All_FU_DNA_Over_All_FU"))
        # FU metrics
        .withColumn("All_FU_NoProc_Over_All_FU", F.when(
            F.col("All_FU") != 0, (F.col("All_FU_NoProc") / F.col("All_FU")) * 100
        ))
        .withColumn("All_FU_Proc_Over_All_FU", F.when(
            F.col("All_FU") != 0, (F.col("All_FU_Proc") / F.col("All_FU")) * 100
        ))
        .withColumn("All_FU_To_All_First", F.when(
            F.col("All_First") != 0, (F.col("All_FU") / F.col("All_First"))
        ))
        # 2WW
        .withColumn("All_2WW_DNA_Over_All_2WW", F.when(
            F.col("All_2WW") != 0, (F.col("All_2WW_DNA") / F.col("All_2WW")) * 100
        ))
        .withColumn("All_First_2WW_DNA_Over_All_First_2WW", F.when(
            F.col("All_First_2WW") != 0, (F.col("All_First_2WW_DNA") / F.col("All_First_2WW")) * 100
        ))
        .withColumn("All_FU_2WW_DNA_Over_All_FU_2WW", F.when(
            F.col("All_FU_2WW") != 0, (F.col("All_FU_2WW_DNA") / F.col("All_FU_2WW")) * 100
        ))
        # Mix shares
        .withColumn("All_FU_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_FU") / F.col("All_Total")) * 100
        ))
        .withColumn("All_First_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_First") / F.col("All_Total")) * 100
        ))
        .withColumn("All_NoProc_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_NoProc") / F.col("All_Total")) * 100
        ))
        .withColumn("All_Proc_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("All_Proc") / F.col("All_Total")) * 100
        ))
        .withColumn("Remote_Total_Over_All_Total", F.when(
            F.col("All_Total") != 0, (F.col("Remote_Total") / F.col("All_Total")) * 100
        ))
        .withColumn("Remote_FU_Over_All_FU", F.when(
            F.col("All_FU") != 0, (F.col("Remote_FU") / F.col("All_FU")) * 100
        ))
        .withColumn("Remote_First_Over_All_First", F.when(
            F.col("All_First") != 0, (F.col("Remote_First") / F.col("All_First")) * 100
        ))
        .withColumn("Remote_DNA_Over_Remote_Total", F.when(
            (F.col("Remote_Total") + F.col("Remote_DNA")) != 0,
            (F.col("Remote_DNA") / (F.col("Remote_Total") + F.col("Remote_DNA"))) * 100
        ))
        .withColumn("F2F_DNA_Over_F2F_Total", F.when(
            (F.col("F2F_Total") + F.col("F2F_DNA")) != 0,
            (F.col("F2F_DNA") / (F.col("F2F_Total") + F.col("F2F_DNA"))) * 100
        ))
    )

# 5. Aggregate to ICB and Region (all orgs included)

df_icb = (
    df_org
    .groupBy("Der_Activity_Month_Date", "ICB", "Treatment_Function_Group")
    .agg(*[F.sum(col(c)).alias(c) for c in count_cols])
)
df_icb = add_rate_metrics(df_icb).withColumn("Level", lit("ICB"))

df_region = (
    df_org
    .groupBy("Der_Activity_Month_Date", "Region_Code", "Treatment_Function_Group")
    .agg(*[F.sum(col(c)).alias(c) for c in count_cols])
)
df_region = add_rate_metrics(df_region).withColumn("Level", lit("Region"))

# 6. Org-level rows

df_org = df_org.withColumn("Level", lit("Org"))

# 7. Combine all levels

final_output = (
    df_org.unionByName(df_icb, allowMissingColumns=True)
          .unionByName(df_region, allowMissingColumns=True)
)

# 8. Final code column

final_output = final_output.withColumn(
    "Adj_Org_Code_Final",
    when(col("Level") == "Org", col("Adj_Org_Code"))
    .when(col("Level") == "ICB", col("ICB"))
    .when(col("Level") == "Region", col("Region_Code"))
)

# 9. Simple - copy and combo columns

copy_map = {
    "All_DNA1": "All_DNA",
    "All_DNA2": "All_DNA",
    "All_FU1": "All_FU",
    "All_FU2": "All_FU",
    "All_FU3": "All_FU",
    "All_FU4": "All_FU",
    "All_FU5": "All_FU",
    "All_First1": "All_First",
    "All_First2": "All_First",
    "All_First3": "All_First",
    "All_Total1": "All_Total",
    "All_Total2": "All_Total",
    "All_Total3": "All_Total",
    "All_Total4": "All_Total",
    "All_Total5": "All_Total",
    "All_Total6": "All_Total",
    "Remote_Total1": "Remote_Total",
    "Remote_Total2": "Remote_Total"
}

for newc, base in copy_map.items():
    if base in final_output.columns:
        final_output = final_output.withColumn(newc, col(base))

combo_pairs = {
    "All_First_plus_All_First_DNA": ("All_First", "All_First_DNA"),
    "All_FU_plus_All_FU_DNA": ("All_FU", "All_FU_DNA"),
    "All_Total_plus_All_DNA": ("All_Total", "All_DNA"),
    "F2F_Total_plus_F2F_DNA": ("F2F_Total", "F2F_DNA"),
    "Remote_Total_plus_Remote_DNA": ("Remote_Total", "Remote_DNA")
}

for newc, (a, b) in combo_pairs.items():
    if a in final_output.columns and b in final_output.columns:
        final_output = final_output.withColumn(newc, col(a).cast("long") + col(b).cast("long"))

# Final preview
display(final_output.limit(20))
print("Container 12 complete —", final_output.count(), "rows")

In [0]:
# 13 – long/skinny OPRT format (Level preserved)
from pyspark.sql.functions import col, lit, explode, array, struct, when
import pyspark.sql.functions as F

# Use the final_output table from container 12 (has Level + Adj_Org_Code_Final)
df_wide = final_output

# Step 1: Drop unnecessary columns
cols_to_drop = [
    "Adj_Org_Code",
    "All_DNA", "All_First", "All_FU",
    "All_Total", "F2F_Total", "Remote_Total"
]
df_wide = df_wide.drop(*[c for c in cols_to_drop if c in df_wide.columns])

# Step 2: Define identifier columns (KEEP Level)
id_cols = [
    "Der_Activity_Month_Date",
    "Region_Code",
    "ICB",
    "Adj_Org_Code_Final",
    "Treatment_Function_Group",
    "Level",
]

# Step 3: Identify metric columns (exclude identifiers)
metric_cols = [c for c in df_wide.columns if c not in id_cols]

# Helper: safe numeric cast under ANSI mode, If value looks like a number → cast to double, Else → NULL

def safe_numeric(colname: str):
    return (
        when(
            col(colname).rlike(r'^[-+]?[0-9]*\.?[0-9]+$'),
            col(colname).cast("double")
        )
        .otherwise(F.lit(None).cast("double"))
        .alias("Metric_Value")
    )

# Step 4: Melt into long/skinny format
opa_oprt_long = (
    df_wide.select(
        *id_cols,
        explode(array(*[
            struct(
                lit(c).alias("OPRT_Metric_Name"),
                safe_numeric(c)
            )
            for c in metric_cols
        ])).alias("kv")
    )
    .select(
        *id_cols,
        col("kv.OPRT_Metric_Name"),
        col("kv.Metric_Value")
    )
)

# Drop null metric rows (zeros are kept)
opa_oprt_long = opa_oprt_long.filter(col("Metric_Value").isNotNull())

display(opa_oprt_long.limit(10))
print(f"Container 13 complete — {opa_oprt_long.count()} rows, {len(opa_oprt_long.columns)} columns")

# Ensures Container 14 uses the cleaned long-format table
final_output = opa_oprt_long


In [0]:
# 14 Prepare Preview Outpatient Metrics and Map Internal IDs
from pyspark.sql.functions import col, concat_ws, lower, regexp_replace, trim
from pyspark.sql.types import StringType, DoubleType, DateType
from pyspark.storagelevel import StorageLevel

# Start from Cell 13 output
df_outpat_metrics = final_output.filter(col("Treatment_Function_Group") != "Other")

# Build combined metric name for joining to ID list
df_outpat_metrics = df_outpat_metrics.withColumn(
    "OPRT_Metric_Name_TFC",
    concat_ws("_", col("OPRT_Metric_Name"), col("Treatment_Function_Group"))
)

# Helper to standardise metric join keys
def normalise_metric_key(col_expr):
    return regexp_replace(
        regexp_replace(
            regexp_replace(
                lower(regexp_replace(trim(col_expr), r"\s+", "_")),
                r"[^a-z0-9_]", ""
            ),
            r"_+", "_"
        ),
        r"^_+|_+$", ""
    )

# Normalise join key on long dataset
df_outpat_metrics = df_outpat_metrics.withColumn(
    "join_metric",
    normalise_metric_key(col("OPRT_Metric_Name_TFC"))
)

# Identify the OPRT TFC metric column in the reference list
metric_list_cols = [c for c in mhs_metric_list.columns if "OPRT" in c and "TFC" in c]
if len(metric_list_cols) == 0:
    raise ValueError(
        "Could not find OPRT TFC metric column name in mhs_metric_list. Review mhs_metric_list columns."
    )
metric_list_col = metric_list_cols[0]

# Prepare cleaned metric lookup
mhs_metric_list_clean = mhs_metric_list.withColumn(
    "join_metric",
    normalise_metric_key(col(metric_list_col))
)

select_cols = ["join_metric", "InternalID"]
if "Description" in mhs_metric_list_clean.columns:
    select_cols.append("Description")
else:
    print("WARNING: mhs_metric_list does not contain a column named 'Description'.")

mhs_metric_list_clean_sel = mhs_metric_list_clean.select(*select_cols).distinct()

# Optional debug join to inspect unmatched metrics
df_outpat_metrics_left = df_outpat_metrics.join(
    mhs_metric_list_clean_sel,
    on="join_metric",
    how="left"
)

unmatched = (
    df_outpat_metrics_left
    .filter(col("InternalID").isNull())
    .select("OPRT_Metric_Name_TFC")
    .distinct()
)

# Final production join (drop unmapped rows)
df_outpat_metrics = (
    df_outpat_metrics
    .join(mhs_metric_list_clean_sel, on="join_metric", how="inner")
    .drop("join_metric")
)

# Filter allowable org codes if reference provided
if "Org_Code" in mhs_allowable_orgs.columns:
    df_outpat_metrics = df_outpat_metrics.join(
        mhs_allowable_orgs.select(col("Org_Code").alias("allowable_code")),
        df_outpat_metrics["Adj_Org_Code_Final"] == col("allowable_code"),
        "inner"
    ).drop("allowable_code")

# Enforce clean data types
df_outpat_metrics = (
    df_outpat_metrics
    .withColumn("Der_Activity_Month_Date", col("Der_Activity_Month_Date").cast(DateType()))
    .withColumn("Adj_Org_Code_Final", col("Adj_Org_Code_Final").cast(StringType()))
    .withColumn("Level", col("Level").cast(StringType()))
    .withColumn("Treatment_Function_Group", col("Treatment_Function_Group").cast(StringType()))
    .withColumn("OPRT_Metric_Name", col("OPRT_Metric_Name").cast(StringType()))
    .withColumn("OPRT_Metric_Name_TFC", col("OPRT_Metric_Name_TFC").cast(StringType()))
    .withColumn("Metric_Value", col("Metric_Value").cast(DoubleType()))
)

# Final output column names
df_outpat_metrics = (
    df_outpat_metrics
    .withColumnRenamed("Der_Activity_Month_Date", "reportingDate")
    .withColumnRenamed("Adj_Org_Code_Final", "providerCode")
    .withColumnRenamed("InternalID", "metricID")
    .withColumnRenamed("Metric_Value", "value")
)

# Keep only required columns
df_outpat_metrics = df_outpat_metrics.select("reportingDate", "providerCode", "metricID", "value")

# Persist once for reuse in later cells
df_outpat_metrics = df_outpat_metrics.persist(StorageLevel.MEMORY_AND_DISK)
_ = df_outpat_metrics.count()

display(df_outpat_metrics.limit(20))


In [0]:
# 15 Extract Allowable OZ Metric Lists and Pull Matching Production Data
import openpyxl
import sys

sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_db_variables import get_mhs_back_end_auto_connection
from DB.db_modes import JDBC_MODES

# Excel source
xlsx_path = "/Workspace/Users/steven.evans4@udal.nhs.uk/MHS-ingestion-for-Outpatients/op_ref.xlsx"
sheet_name = "Lists and df"

groups = [
    ("df_fofu", 4),
    ("df_miss_all", 9),
    ("df_miss_fst", 14),
    ("df_miss_f2f", 19),
    ("df_miss_2ww", 24),
    ("df_fu_nop", 29),
    ("df_remote", 34),
    ("df_all", 39),
]

def extract_metric_id_lists(xlsx_path: str, sheet_name: str, groups, start_row: int = 6):
    wb = openpyxl.load_workbook(xlsx_path, data_only=True)

    if sheet_name not in wb.sheetnames:
        raise ValueError(f"Sheet '{sheet_name}' not found. Available sheets: {wb.sheetnames}")

    ws = wb[sheet_name]
    out = {}

    for name, metric_col in groups:
        ids = []

        for r in range(start_row, ws.max_row + 1):
            v = ws.cell(r, metric_col).value

            if v is None or str(v).strip() == "":
                if ids:
                    lookahead_end = True
                    for rr in range(r, min(r + 6, ws.max_row + 1)):
                        vv = ws.cell(rr, metric_col).value
                        if vv is not None and str(vv).strip() != "":
                            lookahead_end = False
                            break
                    if lookahead_end:
                        break
                continue

            ids.append(str(v).strip())

        seen = set()
        ids = [x for x in ids if not (x in seen or seen.add(x))]

        if len(ids) == 0:
            print(f"WARNING: '{name}' list extracted 0 Metric IDs.")

        out[name] = ids

    return out

metric_lists = extract_metric_id_lists(xlsx_path, sheet_name, groups)

# Combine all OZ codes from the extracted lists
all_metric_ids = []
seen = set()

for ids in metric_lists.values():
    for x in ids:
        if x.startswith("OZ") and x not in seen:
            seen.add(x)
            all_metric_ids.append(x)

print("Number of OZ codes:", len(all_metric_ids))
print("Example codes:", all_metric_ids[:10])

if not all_metric_ids:
    raise ValueError("No OZ metric IDs found in the workbook lists.")

# Build SQL IN clause
metric_id_sql = ", ".join("'" + x.replace("'", "''") + "'" for x in all_metric_ids)

# Connect to SQL Server
mhs_backend_conn = get_mhs_back_end_auto_connection(db_type='production')

# Query Production data
query = f"""
SELECT
       CAST(mv.[reportingDate] AS DATE) AS reportingDate
      ,CAST(mv.[value] AS FLOAT) AS value
      ,m.[InternalID] AS metricID
      ,p.[Code] AS providerCode
FROM [dbo].[MeasureValue] mv
JOIN [dbo].[Measure] m
    ON [MeasureID] = m.ID
LEFT JOIN [dbo].[Provider] p
    ON p.ID = mv.ProviderID
WHERE mv.[reportingDate] > '2023-03-31'
  AND m.[InternalID] IN ({metric_id_sql})
"""

prod_df = mhs_backend_conn.read_from_db(query)

display(prod_df)

prod_row_count = prod_df.count()
display(spark.createDataFrame([(prod_row_count,)], ["row_count"]))

In [0]:
# 16 Build monthly reconciliation buckets using:
# - pre-2024 excluded from change logic
# - latest preview month = full insert
# - latest common month only = compare for changes and deletes

from pyspark.sql import functions as F
from pyspark.sql.types import DateType, StringType, DoubleType
from pyspark.storagelevel import StorageLevel

spark.catalog.clearCache()

# -----------------------------
# Standardise Production
# -----------------------------
prod_compare = (
    prod_df
    .withColumn("reportingDate", F.col("reportingDate").cast(DateType()))
    .withColumn("metricID", F.trim(F.col("metricID")).cast(StringType()))
    .withColumn("providerCode", F.trim(F.col("providerCode")).cast(StringType()))
    .withColumn("value", F.col("value").cast(DoubleType()))
    .select("reportingDate", "metricID", "providerCode", "value")
)

# -----------------------------
# Standardise Preview
# -----------------------------
preview_compare = (
    df_outpat_metrics
    .withColumn("reportingDate", F.col("reportingDate").cast(DateType()))
    .withColumn("metricID", F.trim(F.col("metricID")).cast(StringType()))
    .withColumn("providerCode", F.trim(F.col("providerCode")).cast(StringType()))
    .withColumn("value", F.col("value").cast(DoubleType()))
    .select("reportingDate", "metricID", "providerCode", "value")
)

# -----------------------------
# Restrict reconciliation logic to 2024 onwards
# -----------------------------
cutoff_date = F.to_date(F.lit("2024-01-01"))

prod_compare = prod_compare.filter(F.col("reportingDate") >= cutoff_date)
preview_compare = preview_compare.filter(F.col("reportingDate") >= cutoff_date)

# -----------------------------
# Aggregate to key grain
# -----------------------------
prod_agg = (
    prod_compare
    .groupBy("reportingDate", "metricID", "providerCode")
    .agg(F.sum("value").alias("prod_value"))
)

preview_agg = (
    preview_compare
    .groupBy("reportingDate", "metricID", "providerCode")
    .agg(F.sum("value").alias("preview_value"))
)

# -----------------------------
# Date sets
# -----------------------------
prod_dates_df = prod_agg.select("reportingDate").distinct()
preview_dates_df = preview_agg.select("reportingDate").distinct()

common_dates_df = preview_dates_df.join(prod_dates_df, on="reportingDate", how="inner")
preview_only_dates_df = preview_dates_df.join(prod_dates_df, on="reportingDate", how="left_anti")
prod_only_dates_df = prod_dates_df.join(preview_dates_df, on="reportingDate", how="left_anti")

# -----------------------------
# Identify latest preview month and latest common month
# -----------------------------
latest_preview_date = (
    preview_dates_df
    .agg(F.max("reportingDate").alias("latest_preview_date"))
    .first()["latest_preview_date"]
)

latest_common_date = (
    common_dates_df
    .agg(F.max("reportingDate").alias("latest_common_date"))
    .first()["latest_common_date"]
)

print(f"latest_preview_date = {latest_preview_date}")
print(f"latest_common_date  = {latest_common_date}")

if latest_preview_date is None:
    raise ValueError("No preview reportingDate found after 2024-01-01 filter.")

if latest_common_date is None:
    raise ValueError("No common reportingDate found between preview and prod after 2024-01-01 filter.")

# -----------------------------
# Bucket 1: New month rows
# Full load for latest preview month
# -----------------------------
bucket_1_new_month_df = (
    preview_agg
    .filter(F.col("reportingDate") == F.lit(latest_preview_date))
    .select(
        "reportingDate",
        "providerCode",
        "metricID",
        F.col("preview_value").alias("value")
    )
    .dropDuplicates()
)

# -----------------------------
# Restrict comparison to latest common month only
# -----------------------------
prod_common_latest_df = (
    prod_agg
    .filter(F.col("reportingDate") == F.lit(latest_common_date))
)

preview_common_latest_df = (
    preview_agg
    .filter(F.col("reportingDate") == F.lit(latest_common_date))
)

prod_keys_common_latest_df = (
    prod_common_latest_df
    .select("reportingDate", "metricID", "providerCode")
    .dropDuplicates()
)

preview_keys_common_latest_df = (
    preview_common_latest_df
    .select("reportingDate", "metricID", "providerCode")
    .dropDuplicates()
)

# -----------------------------
# Bucket 2: Matched keys with changed values
# latest common month only
# -----------------------------
matched_keys_common_latest_df = (
    preview_keys_common_latest_df
    .join(
        prod_keys_common_latest_df,
        on=["reportingDate", "metricID", "providerCode"],
        how="inner"
    )
)

matched_value_compare_common_latest_df = (
    matched_keys_common_latest_df
    .join(
        preview_common_latest_df.select(
            "reportingDate", "metricID", "providerCode", "preview_value"
        ),
        on=["reportingDate", "metricID", "providerCode"],
        how="inner"
    )
    .join(
        prod_common_latest_df.select(
            "reportingDate", "metricID", "providerCode", "prod_value"
        ),
        on=["reportingDate", "metricID", "providerCode"],
        how="inner"
    )
    .withColumn("difference_value", F.col("preview_value") - F.col("prod_value"))
)

bucket_2_changed_values_df = (
    matched_value_compare_common_latest_df
    .filter(F.abs(F.col("difference_value")) > 1e-9)
    .select(
        "reportingDate",
        "providerCode",
        "metricID",
        F.col("preview_value").alias("value")
    )
    .dropDuplicates()
)

# -----------------------------
# Bucket 3: Preview-only keys
# latest common month only
# -----------------------------
preview_only_keys_common_latest_df = (
    preview_keys_common_latest_df
    .join(
        prod_keys_common_latest_df,
        on=["reportingDate", "metricID", "providerCode"],
        how="left_anti"
    )
)

bucket_3_preview_only_keys_df = (
    preview_only_keys_common_latest_df
    .join(
        preview_common_latest_df.select(
            "reportingDate", "metricID", "providerCode", "preview_value"
        ),
        on=["reportingDate", "metricID", "providerCode"],
        how="inner"
    )
    .select(
        "reportingDate",
        "providerCode",
        "metricID",
        F.col("preview_value").alias("value")
    )
    .dropDuplicates()
)

# -----------------------------
# Bucket 4 raw: Prod-only keys
# latest common month only
# -----------------------------
prod_only_keys_common_latest_df = (
    prod_keys_common_latest_df
    .join(
        preview_keys_common_latest_df,
        on=["reportingDate", "metricID", "providerCode"],
        how="left_anti"
    )
)

bucket_4_delete_candidates_raw_df = (
    prod_only_keys_common_latest_df
    .withColumn("value", F.lit(None).cast(DoubleType()))
    .select("reportingDate", "providerCode", "metricID", "value")
    .dropDuplicates()
)

# -----------------------------
# Bucket 4 scoped:
# latest common month only, only metric/provider combinations seen in preview that month
# -----------------------------
preview_metric_scope_latest_df = (
    preview_common_latest_df
    .select("metricID")
    .distinct()
)

preview_provider_scope_latest_df = (
    preview_common_latest_df
    .select("providerCode")
    .distinct()
)

prod_keys_common_latest_scoped_df = (
    prod_keys_common_latest_df
    .join(preview_metric_scope_latest_df, on="metricID", how="inner")
    .join(preview_provider_scope_latest_df, on="providerCode", how="inner")
)

prod_only_keys_common_latest_scoped_df = (
    prod_keys_common_latest_scoped_df
    .join(
        preview_keys_common_latest_df,
        on=["reportingDate", "metricID", "providerCode"],
        how="left_anti"
    )
)

bucket_4_delete_candidates_scoped_df = (
    prod_only_keys_common_latest_scoped_df
    .withColumn("value", F.lit(None).cast(DoubleType()))
    .select("reportingDate", "providerCode", "metricID", "value")
    .dropDuplicates()
)

# -----------------------------
# Final stitched outputs
# -----------------------------
final_insert_df = (
    bucket_1_new_month_df
    .unionByName(bucket_2_changed_values_df)
    .unionByName(bucket_3_preview_only_keys_df)
    .dropDuplicates()
)

final_delete_df = bucket_4_delete_candidates_scoped_df.dropDuplicates()

# -----------------------------
# Materialise key outputs
# -----------------------------
prod_agg = prod_agg.persist(StorageLevel.MEMORY_AND_DISK)
preview_agg = preview_agg.persist(StorageLevel.MEMORY_AND_DISK)
prod_keys_common_latest_df = prod_keys_common_latest_df.persist(StorageLevel.MEMORY_AND_DISK)
preview_keys_common_latest_df = preview_keys_common_latest_df.persist(StorageLevel.MEMORY_AND_DISK)
preview_common_latest_df = preview_common_latest_df.persist(StorageLevel.MEMORY_AND_DISK)
final_insert_df = final_insert_df.persist(StorageLevel.MEMORY_AND_DISK)
final_delete_df = final_delete_df.persist(StorageLevel.MEMORY_AND_DISK)

# -----------------------------
# Summary counts
# -----------------------------
counts_data = [
    ("prod_df", prod_df.count()),
    ("prod_agg", prod_agg.count()),
    ("preview_agg", preview_agg.count()),
    ("common_dates_df", common_dates_df.count()),
    ("preview_only_dates_df", preview_only_dates_df.count()),
    ("prod_only_dates_df", prod_only_dates_df.count()),
    ("bucket_1_new_month_df", bucket_1_new_month_df.count()),
    ("bucket_2_changed_values_df", bucket_2_changed_values_df.count()),
    ("bucket_3_preview_only_keys_df", bucket_3_preview_only_keys_df.count()),
    ("bucket_4_delete_candidates_raw_df", bucket_4_delete_candidates_raw_df.count()),
    ("bucket_4_delete_candidates_scoped_df", bucket_4_delete_candidates_scoped_df.count()),
    ("final_insert_df", final_insert_df.count()),
    ("final_delete_df", final_delete_df.count())
]

counts_df = spark.createDataFrame(counts_data, ["dataset", "row_count"])
display(counts_df)

In [0]:
# 17 QA summaries for monthly-window logic
from pyspark.sql import functions as F

print(f"Bucket 1 - New month rows: {bucket_1_new_month_df.count()}")
print(f"Bucket 2 - Changed matched values on latest common month: {bucket_2_changed_values_df.count()}")
print(f"Bucket 3 - Preview-only keys on latest common month: {bucket_3_preview_only_keys_df.count()}")
print(f"Bucket 4 raw - Prod-only keys on latest common month: {bucket_4_delete_candidates_raw_df.count()}")
print(f"Bucket 4 scoped - Prod-only keys on latest common month within preview metric/provider scope: {bucket_4_delete_candidates_scoped_df.count()}")
print(f"Final insert/update row count: {final_insert_df.count()}")
print(f"Final delete row count: {final_delete_df.count()}")

insert_by_date_df = (
    final_insert_df
    .groupBy("reportingDate")
    .agg(F.count("*").alias("row_count"))
    .orderBy("reportingDate")
)

delete_by_date_df = (
    final_delete_df
    .groupBy("reportingDate")
    .agg(F.count("*").alias("row_count"))
    .orderBy("reportingDate")
)

display(insert_by_date_df)
display(delete_by_date_df)


In [0]:
# 18 DELETE dry-run for monthly-window logic
from pyspark.sql import functions as F

EXPECTED_MIN_DELETE_ROWS = 200
EXPECTED_MAX_DELETE_ROWS = 250

def main():
    delete_df = (
        final_delete_df
        .select("reportingDate", "providerCode", "metricID", "value")
        .dropDuplicates()
        .persist()
    )

    delete_count = delete_df.count()

    print(f"Delete candidate row count: {delete_count}")

    if delete_count == 0:
        print("No delete candidates found.")
    elif EXPECTED_MIN_DELETE_ROWS <= delete_count <= EXPECTED_MAX_DELETE_ROWS:
        print(
            f"Delete count is within expected range "
            f"({EXPECTED_MIN_DELETE_ROWS}-{EXPECTED_MAX_DELETE_ROWS})."
        )
    else:
        print(
            f"WARNING: Delete count {delete_count} is outside expected range "
            f"({EXPECTED_MIN_DELETE_ROWS}-{EXPECTED_MAX_DELETE_ROWS})."
        )

    display(delete_df.orderBy("reportingDate", "metricID", "providerCode"))

if __name__ == "__main__":
    main()

In [0]:
# 19 Diagnose the 632 delete rows
from pyspark.sql import functions as F

print(f"latest_common_date = {latest_common_date}")
print(f"final_delete_count = {final_delete_df.count()}")

delete_by_metric_df = (
    final_delete_df
    .groupBy("metricID")
    .agg(F.count("*").alias("row_count"))
    .orderBy(F.desc("row_count"), "metricID")
)

delete_by_provider_df = (
    final_delete_df
    .groupBy("providerCode")
    .agg(F.count("*").alias("row_count"))
    .orderBy(F.desc("row_count"), "providerCode")
)

delete_by_metric_provider_df = (
    final_delete_df
    .groupBy("metricID", "providerCode")
    .agg(F.count("*").alias("row_count"))
    .orderBy(F.desc("row_count"), "metricID", "providerCode")
)

display(delete_by_metric_df)
display(delete_by_provider_df)
display(delete_by_metric_provider_df)
display(final_delete_df.orderBy("metricID", "providerCode"))

In [0]:
# 20 Latest common month delete diagnostics
from pyspark.sql import functions as F

delete_latest_by_metric_df = (
    final_delete_df
    .groupBy("metricID")
    .agg(F.count("*").alias("row_count"))
    .orderBy(F.desc("row_count"), "metricID")
)

delete_latest_by_provider_df = (
    final_delete_df
    .groupBy("providerCode")
    .agg(F.count("*").alias("row_count"))
    .orderBy(F.desc("row_count"), "providerCode")
)

print(f"latest_common_date = {latest_common_date}")
print(f"final_delete_count = {final_delete_df.count()}")

display(delete_latest_by_metric_df)
display(delete_latest_by_provider_df)
display(final_delete_df.orderBy("metricID", "providerCode"))

In [0]:
# 21 QA CSV exports
from pyspark.sql import functions as F

base_path = "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/ElectiveRecovery/Projects"

bucket_1_new_month_df.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_QA_bucket_1_new_month_df.csv")

bucket_2_changed_values_df.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_QA_bucket_2_changed_values_latest_common_month_df.csv")

bucket_3_preview_only_keys_df.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_QA_bucket_3_preview_only_keys_latest_common_month_df.csv")

bucket_4_delete_candidates_raw_df.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_QA_bucket_4_delete_candidates_raw_latest_common_month_df.csv")

bucket_4_delete_candidates_scoped_df.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_QA_bucket_4_delete_candidates_scoped_latest_common_month_df.csv")

final_insert_df.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_QA_final_insert_df_monthly_window.csv")

final_delete_df.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save(f"{base_path}/OP_QA_final_delete_df_monthly_window.csv")

print("QA CSV exports completed.")

In [0]:

%skip
# 18 Ingest changed Preview rows to MHS using INSERT
import sys
from pyspark.sql import functions as F

sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_import import MHS_IngestionHub
from MHS_Helper.mhs_db_config import INSERT

RUN_INSERT = False

def upload_to_mhs_insert(dfih, desc, skip_check: bool):
    sdf = (
        dfih
        .withColumn("reportingDate", F.to_date("reportingDate"))
        .select("reportingDate", "providerCode", "metricID", "value")
        .dropDuplicates()
    )

    MHS_IngestionHub.upload_many(
        mhs_df=sdf,
        description=desc,
        loaded_by="steven.evans4@nhs.net",
        mhs_mode=INSERT,
        skip_existing_data_check=skip_check
    )

def main():
    insert_df = (
        non_zero_differences_df
        .filter(F.abs(F.col("preview_value")) > 1e-9)
        .select(
            "reportingDate",
            "providerCode",
            "metricID",
            F.col("preview_value").alias("value")
        )
        .dropDuplicates()
        .persist()
    )

    row_count = insert_df.count()

    if row_count == 0:
        print("No changed preview rows found for insert — skipping upload.")
        return

    max_date = insert_df.agg(F.max("reportingDate").alias("max_date")).collect()[0]["max_date"]

    try:
        max_date_str = max_date.strftime("%Y-%m-%d")
    except Exception:
        max_date_str = str(max_date)

    desc = f"df_outpat_metrics_insert_changes_{max_date_str}"

    print(f"INSERT description: {desc}")
    print(f"Row count to insert: {row_count}")

    display(insert_df.orderBy("metricID", "providerCode", "reportingDate"))

    if RUN_INSERT:
        print("RUN_INSERT = True, proceeding with INSERT upload...")
        upload_to_mhs_insert(insert_df, desc, skip_check=True)
        print("INSERT upload completed successfully.")
    else:
        print("RUN_INSERT = False, so this is a dry run only. No insert has been executed.")

if __name__ == "__main__":
    main()

In [0]:

%skip
# 19 Controlled DELETE test for rows in PROD but not in PREVIEW
import sys
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

sys.path.append('/Workspace/Repos/MHS-analytics/MHS-analytics/Python_Packages')

from MHS_Helper.mhs_import import MHS_IngestionHub
from MHS_Helper.mhs_db_config import DELETE

# Safety switches
RUN_DELETE = False
EXPECTED_MIN_DELETE_ROWS = 200
EXPECTED_MAX_DELETE_ROWS = 250

def upload_to_mhs_delete(dfih, desc, skip_check: bool):
    sdf = (
        dfih
        .withColumn("reportingDate", F.to_date("reportingDate"))
        .withColumn("value", F.col("value").cast(DoubleType()))
        .select("reportingDate", "providerCode", "metricID", "value")
        .dropDuplicates()
    )

    MHS_IngestionHub.upload_many(
        mhs_df=sdf,
        description=desc,
        loaded_by="steven.evans4@nhs.net",
        mhs_mode=DELETE,
        skip_existing_data_check=True
    )

def main():
    delete_df = (
        delete_candidates_df
        .select("reportingDate", "providerCode", "metricID", "value")
        .dropDuplicates()
        .persist()
    )

    delete_count = delete_df.count()

    print(f"Delete candidate row count: {delete_count}")

    display(delete_df.orderBy("metricID", "providerCode", "reportingDate"))

    if delete_count == 0:
        print("No delete candidates found — skipping delete step.")
        return

    if not (EXPECTED_MIN_DELETE_ROWS <= delete_count <= EXPECTED_MAX_DELETE_ROWS):
        print(
            f"Delete count {delete_count} is outside expected range "
            f"{EXPECTED_MIN_DELETE_ROWS}-{EXPECTED_MAX_DELETE_ROWS}. "
            f"Delete blocked for safety."
        )
        return

    max_date = delete_df.agg(F.max("reportingDate").alias("max_date")).collect()[0]["max_date"]

    try:
        max_date_str = max_date.strftime("%Y-%m-%d")
    except Exception:
        max_date_str = str(max_date)

    desc = f"df_outpat_metrics_delete_missing_from_preview_{max_date_str}"

    print(f"DELETE description: {desc}")
    print("Safety check passed.")

    if RUN_DELETE:
        print("RUN_DELETE = True, proceeding with DELETE upload...")
        upload_to_mhs_delete(delete_df, desc, skip_check=True)
        print("DELETE upload completed successfully.")
    else:
        print("RUN_DELETE = False, so this is a dry run only. No delete has been executed.")

if __name__ == "__main__":
    main()